### Tutorial 4 — Patch clustering on cHL with KRONOS2 + Leiden

Patches with similar tissue composition should be close to each other in embedding space. This tutorial takes the cHL CODEX slide you ingested in
[Tutorial 1](1-Tissue-Ingest.ipynb) and turns its **KRONOS2 patch embeddings** into **spatial domains** in unsupervised manner:

1. **Marker panel** — restrict extraction to the phenotypic markers used for phenotyping
2. **Patch extraction** — a tissue-filtered grid over the slide
3. **KRONOS2 features on the subset** — a 768-dim embedding per patch, with each marker z-scored by KRONOS2's **own stored mean/std**
4. **Leiden clustering** (resolution 0.5) — group patches into tissue phenotypes
5. **Cluster overlay** — paint the clusters back onto a downsampled view of the raw slide

> **Prerequisites.** Run [Tutorial 0](0-Example-Data-Download.ipynb) (data), [Tutorial 1](1-Tissue-Ingest.ipynb) (ingest), and [Tutorial 2](2-Step-by-Step-Patch-Feature-Extraction.ipynb) (tissue + patches) first. Tutorial 4 reopens the slide store from Tutorial 1 and reuses the tissue mask from Tutorial 2 to keep patches on-tissue. **Launch Jupyter from the `tutorials/` directory**, and run the `coral` commands below from a terminal at the repo root (`ESB-internal/`).

The CORAL steps (panel → patches → features) are shown as **`coral` CLI commands** — run them in a terminal, exactly as in Tutorials 1–2 (the Python equivalents were covered there). The CLI runs each step over a **whole cohort** — every `.zarr` store in a **job directory** (here `tutorials/example-data/processed`, the folder Tutorial 1 wrote `raw_image.zarr` into). Leiden clustering and the overlay live **downstream of CORAL** (scanpy + matplotlib); **Section 5 opens the slide in Python** and takes over from there.

| Step               | CLI                                  | Python                                          |
| ------------------ | ------------------------------------ | ----------------------------------------------- |
| Patches            | `coral patch`                          | `slide.extract_patches(cfg)`                    |
| Features (subset)  | `coral extract --subset panel.yaml`    | `slide.encode_features(k, cfg, channels=panel)` |
| Cluster + overlay  | *(downstream — not a CORAL command)*  | *(scanpy + matplotlib, below)*                  |

#### 0 — Installation

```bash
# KRONOS2 foundation-model features (needs a CUDA GPU) — for step 3
uv sync --extra kronos2

# Leiden clustering + UMAP live downstream of CORAL, so they are not CORAL
# dependencies — install them into the tutorial environment directly:
uv pip install scanpy igraph leidenalg umap-learn
```

`scanpy` brings `anndata`, the neighbour graph, and Leiden; `igraph` + `leidenalg` are the Leiden backend; `umap-learn` is only for the optional 2-D embedding plot. None of them are CORAL dependencies — clustering is a downstream step CORAL deliberately leaves to the consumer.

#### 1. Reopen the slide from Tutorial 1

Phenotyping builds on the **same slide store** Tutorial 1 wrote — no re-ingest. `convert_to_canonical` named that store after its input folder (`raw_image/` → `raw_image.zarr`), which Section 5 reopens to start clustering. Tutorial 2 ran tissue detection on this store, which we rely on to keep patches on-tissue.

#### 2. Choose the phenotyping marker panel

Phenotyping doesn't need every channel — it needs the **phenotypic markers** that separate tissue compartments (immune, stromal, epithelial, vascular).

CORAL canonicalises marker names at ingest, so a couple of raw CODEX names land on cleaner registry names — the two you fixed in Tutorial 1's marker map:

- `DAPI-01` → **`dapi`** (the nuclear stain)
- `CYTOKERITIN` → **`cytokeratin`**

So we name the panel in CORAL's **canonical, lower-cased** vocabulary (what `slide.markers` prints). A `Selection` (the same object `coral ingest --subset` uses) turns that list into a marker filter for extraction — matched case-insensitively, so `dapi` selects the `DAPI` channel.

**CLI — the panel as a subset YAML.** The CLI expresses the same panel as a small `--subset` file — the identical `channels` glob the `Selection` uses. Save it as `panel.yaml`:

```yaml
name: chl_phenotyping
channels:
  include: [dapi, cd11b, cd11c, cd15, cd163, cd20, cd206, cd30, cd31,
            cd4, cd56, cd68, cd7, cd8, cytokeratin, foxp3, mct, podoplanin]
```

#### 3. Patch extraction

`extract_patches` tiles the slide into a tissue-filtered grid at the base resolution and writes `patches/<slug>/`. We use a **64 px** patch (~24 µm here) — a fine grid so the cluster map resolves tissue structure. This is a **new** patch set (different size from Tutorial 2's 256 px grid), so it creates a new slug (`0.37mpp_64px`).

> We use `overlap=0.0` for computational efficiency. For smoother overlay, please use non-zero overlap (e.g., `overlap=0.5`).

**CLI** The same 64 px tissue-filtered grid on every store in the job directory:

```bash
uv run coral patch --job-dir tutorials/example-data/processed --patch-size 64
```

#### 4. KRONOS2 features on the marker subset

Now extract a KRONOS2 embedding for each patch — but **only over the panel**. The `--subset panel.yaml` selection restricts extraction to those 18 markers; KRONOS2 is marker-aware, so the embedding reflects exactly the channels it saw.

##### KRONOS2 applies each marker's mean/std for you

**KRONOS2 carries those statistics inside the checkpoint** and applies them right before the ViT — so the marker-wise z-score happens automatically during extraction, bit-exact, with no CSV to manage.

The one thing worth checking is that every panel marker is actually **in** KRONOS2's stats table; any marker that isn't silently falls back to a default `(mean, std)`. For this panel all 18 z-score with their own learned statistics.

**CLI — KRONOS2 on the subset.** Extract over the panel from `panel.yaml` (GPU + `kronos2` extra), selecting the patch set by its slug `0.37mpp_64px`:

```bash
uv run coral extract --job-dir tutorials/example-data/processed --extractor KRONOS2 --patches 0.37mpp_64px --subset panel.yaml --batch-size 16 --gpu 0
```

The subset filename names the output variant (`markers_panel`), so it sits beside any all-marker run.

#### 5. Leiden clustering (resolution 0.5)

The embeddings are now an `(n_patches, 768)` matrix — standard input to community detection. We build a k-nearest-neighbour graph on the embeddings and run **Leiden** at **resolution 0.5** (higher resolution → more, smaller clusters). Each patch gets an integer cluster id — its tissue phenotype. Alternatively, we can also use K-means clustering for simplicity.

We use `scanpy` (it wraps the neighbour graph + Leiden), pinning `random_state` so the clustering is reproducible.

##### Alternatively: load features extracted on another machine
(THIS MIGHT BE REMOVED from ANDREW)
Feature extraction (step 4) is the one step that needs a GPU. If you ran it elsewhere, e.g., a GPU box, you don't need to repeat it here: **copy the extracted features into this slide store and read them back**, with no GPU and without the `kronos2` extra.

CORAL writes every extraction under `features/<patch-slug>/<encoder>/<variant>/` inside the `.zarr` store (the slug is `<mpp>mpp_<size>px`; the variant is named after the `--subset` file, here `markers_panel`, or `markers_all` with no subset). For this run that is:

```
raw_image.zarr/
├── patches/0.37mpp_64px/                       # patch coords (step 3)
└── features/0.37mpp_64px/KRONOS2/markers_panel/  # 768-d embeddings (step 4)
```

Copy **both** folders from the machine that ran the extraction into your local store. The `patches/<slug>/` folder matters: the feature store doesn't duplicate patch positions — `slide.features(...)` joins each patch's `x`/`y` from `patches/<slug>/coords`, and the overlay (step 6) needs them.

Then load the features by **name** — passing the string `"KRONOS2"` reads the stored array directly, so nothing loads the model. Run this cell **instead of** step 4's extraction; it defines the same `emb` the clustering below expects.

In [ ]:
from pathlib import Path

from coral import CoralSlide
from coral.config import PatchConfig
from coral.config.subset import Selection

# Run this notebook from the tutorials/ directory. Tutorial 1 ingested the
# slide; the CLI steps above (Tutorials 1-2 patterns) produced the KRONOS2
# features. Reopen the slide and load those embeddings by NAME — reading the
# stored array, so nothing loads the model (no GPU, no kronos2 extra).
OUTPUT_DIR = Path("example-data/processed")
PATCH_SIZE = 64          # ~24 um at 0.37 mpp — the fine grid from step 3
LEIDEN_RESOLUTION = 0.5  # higher -> more, smaller clusters
SEED = 0                 # pin every random step (neighbours, Leiden, UMAP)

# The phenotyping panel (same 18 markers as panel.yaml above).
PANEL = [
    "dapi", "cd11b", "cd11c", "cd15", "cd163", "cd20", "cd206", "cd30",
    "cd31", "cd4", "cd56", "cd68", "cd7", "cd8", "cytokeratin", "foxp3",
    "mct", "podoplanin",
]
panel = Selection(include=PANEL)  # the panel markers.yaml selected at extract

slide = CoralSlide.open(OUTPUT_DIR / "raw_image.zarr")
cfg = PatchConfig(patch_size=PATCH_SIZE, overlap=0.0)

# The CLI named the extraction variant after panel.yaml -> markers_panel;
# read it back by that variant name (suffix), not by re-passing the channels.
emb = slide.features("KRONOS2", cfg, suffix="markers_panel")
print("KRONOS2 features:", emb["features"].shape, "(n_patches, 768)")
emb

In [ ]:
import anndata as ad
import numpy as np
import scanpy as sc

X = np.asarray(emb["features"].values, dtype=np.float32)  # (n_patches, 768)
adata = ad.AnnData(X)

# Graph on the raw 768-d embeddings (use_rep="X" -> no intermediate PCA).
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X", random_state=SEED)
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    random_state=SEED,
    key_added="leiden",
    flavor="igraph",
    n_iterations=2,
    directed=False,
)

labels = adata.obs["leiden"].astype(int).to_numpy()
n_clusters = int(labels.max()) + 1
print(f"{n_clusters} clusters at resolution {LEIDEN_RESOLUTION}")
print("patches per cluster:")
print(adata.obs["leiden"].value_counts().sort_index())

#### 6. Cluster overlay on the (downsampled) raw slide

Each patch carries its level-0 `(x, y)` top-left (joined into `emb` above), so we can paint every patch's cluster id back onto the slide's footprint. A whole CODEX slide is far too big to raster at full resolution, so we build the map — and read the raw nuclear channel — **downsampled** by an integer factor chosen to bring the long side to ~1500 px.

Three panels: the downsampled nuclear backdrop, the cluster map, and the two blended — the cluster overlay on the raw image.

> CORAL's overlay writers (tissue, patches) render their PNGs headlessly by switching matplotlib to the non-interactive **Agg** backend, which silences inline plots afterwards. We re-enable inline rendering with `%matplotlib inline` before drawing — one line at the top of each plot cell keeps it robust even if you re-run the patch step.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from coral.viz import normalize_uint8

xs = emb.coords["x"].values.astype(int)   # patch top-left x, level 0
ys = emb.coords["y"].values.astype(int)   # patch top-left y, level 0
size = int(cfg.patch_size)

_, H, W = slide.image.shape
factor = max(1, int(np.ceil(max(H, W) / 1500)))   # long side -> ~1500 px
Hs, Ws = int(np.ceil(H / factor)), int(np.ceil(W / factor))

# Cluster map at the downsampled scale; NaN where no patch (stays transparent).
cluster_map = np.full((Hs, Ws), np.nan, dtype=float)
for x, y, lab in zip(xs, ys, labels):
    cluster_map[y // factor:(y + size) // factor,
                x // factor:(x + size) // factor] = lab

# Downsampled nuclear backdrop — a lazy strided read, so it never
# materialises the full-resolution channel.
nuc = slide.nuclear_channel
backdrop = normalize_uint8(np.asarray(slide.image[nuc, ::factor, ::factor]))

vmax = max(n_clusters - 1, 1)
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(backdrop, cmap="gray")
ax[0].set_title(f"Nuclear (DAPI), downsampled {factor}x")
ax[1].imshow(cluster_map, cmap="tab20", vmin=0, vmax=vmax, interpolation="nearest")
ax[1].set_title(f"Leiden clusters (res {LEIDEN_RESOLUTION}, k={n_clusters})")
ax[2].imshow(backdrop, cmap="gray")
ax[2].imshow(cluster_map, cmap="tab20", vmin=0, vmax=vmax, alpha=0.55,
             interpolation="nearest")
ax[2].set_title("Cluster overlay on raw image")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

##### Optional — a 2-D map of the embedding

A UMAP of the KRONOS2 embeddings, coloured by Leiden cluster, shows the phenotype structure the graph clustered on (well-separated blobs ⇒ crisp phenotypes).

In [ ]:
%matplotlib inline
sc.tl.umap(adata, random_state=SEED)
sc.pl.umap(adata, color="leiden", title=f"KRONOS2 patches — Leiden res {LEIDEN_RESOLUTION}")

#### 7. Recap

From the slide you ingested in Tutorial 1 you restricted to a phenotyping panel, cut a fine tissue-filtered grid, embedded each patch with KRONOS2 (each marker z-scored by KRONOS2's own stored mean/std), clustered the embeddings with Leiden at resolution 0.5, and painted the phenotypes back onto the raw slide — every CORAL step tracked in the slide's state file:

In [ ]:
{step: info["status"] for step, info in slide.status().items()}

**Where next?**

- **Name the phenotypes.** The cluster × marker signature turns each unsupervised cluster into a biological label — B-cell zone, tumour epithelium, vasculature, and so on.
- **Resolution sweep.** Re-run Leiden at a few resolutions (0.3–1.0) to move between coarse tissue compartments and fine niches; the embeddings don't change, only the clustering.
- **Cohorts.** Point the `coral` CLI at a directory of slides (`coral patch`, `coral extract --subset panel.yaml`) for parallel, resumable, per-slide KRONOS2 extraction, then cluster the pooled embeddings to phenotype consistently across the cohort.
- **Cells.** For a per-cell rather than per-region readout, [Tutorial 3](3-Cell-Segmentation-and-Feature-Extraction.ipynb) runs the same KRONOS2 encoder on cell-centered patches.